Script générant un attaque adverse simple : rotation. On test soit :

- Tout les angles pour toutes les images pour obtenir une accuracy moyenne du réseaux sur un data set en fonction de l'angle
- Tout les angle pour chaque image pour un label donné pour obtenir les angles maximisant ou minimisant la prediction d'un réseau sur ce label

--- TODO : ---

> Simplifier un peu le code et l'adapter au nouveaux réseaux 


In [ ]:
from retinotopy import *

In [ ]:
args = Params()
args.folders = ['val']
args.do_rotation = True
args

# rotating a whole dataset by 45° :

In [ ]:
data_set_types = ['full', 'boxes', 'focus', ]

In [ ]:

# for data_set_type in data_set_types:
#     print(50*'=')
#     print(f'{data_set_type=}')
#     args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images to perform the training
#     print(50*'-')

#     for args.do_polar in [True, False]:
#         print(f'*** {args.do_polar=}')
#         dataloaders = datasets_transforms(args, angle_min=44, angle_max=46)
#         for _, (images, labels) in enumerate(dataloaders['val']):
#             images, labels = images.to('cpu'), labels.to('cpu')
#             break
#         imshow(images[:5], title='Example of cartesian input images' if not(args.do_polar) else 'Example of log-polar input images', fig_height=5)
#         plt.show()
#     print(50*'=')

# testing each network for different rotations

In [ ]:
%ls {data_cache}/*results_test-rotations.json

In [ ]:
# %rm cached_data/2024-04-25_results_test-rotations.json

In [ ]:
angles = np.linspace(-180, 180, 24, endpoint=False)

In [ ]:
delta_angle = angles[1] - angles[0]
angles_min = angles - delta_angle/2
angles_max = angles + delta_angle/2

# Loading and testing the networks
for data_set_type in data_set_types:
    
    print(50*'=')
    print(f'{data_set_type=}')
    args = Params()
    args.folders = ['val']
    args.do_rotation = True
    args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images to perform the training
    print(50*'-')
        
    for model_name, lw in  zip(['resnet101', 'resnet50', 'resnet18'], [4, 2, 1]): #, 
        print(f'{model_name=}')

        for do_polar, color in zip([True, False], ['b', 'r']):
            args.do_polar = do_polar
            print(f'{args.do_polar=}')
            print(50*'.')
            df_filename = f'{data_cache}/{datetag}_{data_set_type}_{model_name}_{do_polar=}_results_test-rotations.json'
            if os.path.isfile(df_filename):
                df_angle = pd.read_json(df_filename)
            else:
                results = np.zeros((0, 3))
                model_filename = f'{data_cache}/{datetag}_{data_set_type}_{model_name}_{do_polar=}.pt'

                if os.path.isfile(model_filename):
                    print(f"Loading pre-trained resnet {model_filename}")
                    model = charge_model(model_filename, do_polar=args.do_polar).to(device).eval()

                    for angle, angle_min, angle_max in zip(angles, angles_min, angles_max):
                        print(f'{angle=}', end='')
                        dataloaders = datasets_transforms(args, angle_min=angle_min, angle_max=angle_max, shuffle=False, verbose=False)
                        with torch.no_grad():
                            for i_image, (images, labels) in enumerate(dataloaders['val']):
                                images, labels = images.to(device), labels.to(device)

                                outputs = model(images)

                                _, preds = torch.max(outputs.data, dim=1)
                                detect = (preds == labels.data).cpu().numpy()

                                results = np.vstack((results, np.vstack([i_image*args.batch_size_val + np.arange(len(detect)), angle*np.ones(len(detect)), detect]).T))                        
                        print(f', Accuracy={detect.mean():.3f}')
                    df_angle = pd.DataFrame(results, columns=['i_image', 'angle', 'match'])     
                    df_angle.to_json(df_filename)                        
                    print(25*'. ')

    print(50*'=')

In [ ]:
df_angle['match'].mean()


In [ ]:
results

In [ ]:
df_angle

In [ ]:
for angle in angles:   
    print(f"{angle=}, accuracy={df_angle[df_angle['angle'] == angle]['match'].mean():.3f}")

In [ ]:
df_angle['match'].mean()

In [ ]:
np.hstack((angles, 180))

In [ ]:
        
for model_name in  ['resnet50', 'resnet18', 'resnet101', ]:
    print(50*'=')
    print(f'{model_name=}')
    fig, ax = plt.subplots(figsize=(fig_width*phi/3, fig_width/phi/2))
    for data_set_type in data_set_types:    
        print(f'{data_set_type=}')
        print(50*'-')

        for do_polar, color in zip([True, False], ['b', 'r']):

            df_filename = f'{data_cache}/{datetag}_{data_set_type}_{model_name}_{do_polar=}_results_test-rotations.json'

            if os.path.isfile(df_filename):
                df_angle = pd.read_json(df_filename)

                results = [df_angle[df_angle['angle'] == angle]['match'].mean() for angle in np.hstack((angles, -180.))]
                label = 'Retino' if do_polar else 'Cartesian'
                label += f' on {data_set_type}'
                ax.plot(np.hstack((angles, 180)), results, color=color, label=label)
        
    #ax.hlines(xmin=-185, xmax=185, y=1/2, ls='--', ec='gray')
    ax.tick_params(axis='x', labelsize=14)
    ax.tick_params(axis='y', labelsize=14)
    ax.set_xlim(-5, 185)
    ax.set_ylim(0, 1)

    for angle in [-180, -90, 0, 90, 180]:
        ax.axvline(x=angle, c='k', ls='--', lw=1)
    ax.set_xticks([-180, -90, 0, 90, 180])
    #ax.set_yscale("logit", use_overline=True) #one_half="1/2", 
    #ax.set_yticks([.7, .75, .8])
    ax.set_ylabel('Average Accuracy', font=font)
    ax.set_xlabel('Rotation angle (°)', font=font)
    plt.legend(bbox_to_anchor=(0.8, 1), loc='upper center', fontsize=10, edgecolor='none')
    plt.tight_layout()
    # plt.xticks(font=font)
    # plt.yticks(font=font);
    plt.show()

In [ ]:
for ext in ['pdf']:
    # fig.savefig(os.path.join('cached_data', f'attack_rotation_imagenet.{ext}'), **opts_savefig)
    fig.savefig(f'attack_rotation_imagenet.{ext}', **opts_savefig)